In [64]:
import pandas as pd 

In [65]:
df = pd.read_csv('Data/hotel_bookings.csv')
df['country'] = df['country'].fillna('Unknown') 
df['children'] = df['children'].fillna(0)
df['agent'] = df['agent'].fillna(0)
df.drop(columns=['company'], axis=1, inplace=True)
df.drop_duplicates(inplace=True)
df


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,3,No Deposit,0.0,0,Transient,0.00,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,4,No Deposit,0.0,0,Transient,0.00,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,No Deposit,0.0,0,Transient,75.00,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,No Deposit,304.0,0,Transient,75.00,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,No Deposit,240.0,0,Transient,98.00,0,1,Check-Out,2015-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,0,No Deposit,394.0,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,0,No Deposit,9.0,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,0,No Deposit,9.0,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,0,No Deposit,89.0,0,Transient,104.40,0,0,Check-Out,2017-09-07


In [66]:
# Total malam setiap minggu
df['total_nights'] = df['stays_in_week_nights']+ df['stays_in_weekend_nights']

In [67]:
df['customers'] = df['adults'] + df['children'] + df['babies']

---------------------

In [68]:
# Mengubah 0 & 1 menjadi No & Yes
df['is_repeated_category'] = df['is_repeated_guest'].apply(lambda x: 'Yes' if x == 1 else 'No' )

In [69]:
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])

In [70]:
# Mengubah nama bulan menjadi angka
month_mapping = {
    'January': 1,
    'February': 2,
    'March': 3,
    'April': 4,
    'May': 5,
    'June': 6,
    'July': 7,
    'August': 8,
    'September': 9,
    'October': 10,
    'November': 11,
    'December': 12
}

df['arrival_date_month'] = df['arrival_date_month'].map(month_mapping)

# Advanced Data Preprocessing

In [71]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.to_list()
numerical_cols

['is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'booking_changes',
 'agent',
 'days_in_waiting_list',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'total_nights',
 'customers']

In [72]:
# Membuat iqr (inter quartile range)
def remove_outliers_iqr(df, columns, x):
    df_clean = df.copy()
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - x * IQR # 1.5
        upper_bound = Q3 + x * IQR # 1.5
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    return df_clean

In [73]:
# lower bound & apper bound

In [74]:
df_no_outliers = remove_outliers_iqr(df, numerical_cols, 2)
df_no_outliers

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_nights,customers,is_repeated_category
4,Resort Hotel,0,14,2015,7,27,1,0,2,2,...,0,Transient,98.00,0,1,Check-Out,2015-07-03,2,2.0,No
6,Resort Hotel,0,0,2015,7,27,1,0,2,2,...,0,Transient,107.00,0,0,Check-Out,2015-07-03,2,2.0,No
7,Resort Hotel,0,9,2015,7,27,1,0,2,2,...,0,Transient,103.00,0,1,Check-Out,2015-07-03,2,2.0,No
8,Resort Hotel,1,85,2015,7,27,1,0,3,2,...,0,Transient,82.00,0,1,Canceled,2015-05-06,3,2.0,No
9,Resort Hotel,1,75,2015,7,27,1,0,3,2,...,0,Transient,105.50,0,0,Canceled,2015-04-22,3,2.0,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119383,City Hotel,0,164,2017,8,35,31,2,4,2,...,0,Transient,87.60,0,0,Check-Out,2017-09-06,6,2.0,No
119384,City Hotel,0,21,2017,8,35,30,2,5,2,...,0,Transient,96.14,0,2,Check-Out,2017-09-06,7,2.0,No
119385,City Hotel,0,23,2017,8,35,30,2,5,2,...,0,Transient,96.14,0,0,Check-Out,2017-09-06,7,2.0,No
119388,City Hotel,0,109,2017,8,35,31,2,5,2,...,0,Transient,104.40,0,0,Check-Out,2017-09-07,7,2.0,No


## Normalisasi
- Min Max Scalling (ada di rentang 0 - 1 saja)
- 

In [75]:
from sklearn.preprocessing import MinMaxScaler

In [76]:
df_minmax_scaled = df_no_outliers.copy()

In [ ]:
min_max_scaled = MinMaxScaler()

In [ ]:
df_minmax_scaled[numerical_cols] = min_max_scaled.fit_transform(df_no_outliers[numerical_cols])

In [79]:
df_minmax_scaled.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,previous_bookings_not_canceled,booking_changes,agent,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date,total_nights,customers
count,41264.000000,41264.000000,41264.000000,41264.000000,41264.000000,41264.000000,41264.000000,41264.000000,41264.0,41264.0,...,41264.0,41264.0,41264.000000,41264.0,41264.000000,41264.0,41264.000000,41264,41264.000000,41264.0
mean,0.313978,0.229539,0.612313,0.492368,0.491431,0.491466,0.254901,0.289712,0.0,0.0,...,0.0,0.0,0.149445,0.0,0.440982,0.0,0.233456,2016-09-01 10:09:47.669639168,0.329728,0.0
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,2014-11-18 00:00:00,0.000000,0.0
25%,0.000000,0.050992,0.500000,0.272727,0.288462,0.233333,0.000000,0.111111,0.0,0.0,...,0.0,0.0,0.016949,0.0,0.328947,0.0,0.000000,2016-03-22 00:00:00,0.181818,0.0
50%,0.000000,0.161473,0.500000,0.454545,0.500000,0.500000,0.250000,0.222222,0.0,0.0,...,0.0,0.0,0.016949,0.0,0.422368,0.0,0.333333,2016-09-09 00:00:00,0.272727,0.0
75%,1.000000,0.354108,1.000000,0.727273,0.692308,0.766667,0.500000,0.444444,0.0,0.0,...,0.0,0.0,0.333333,0.0,0.548246,0.0,0.333333,2017-03-06 00:00:00,0.454545,0.0
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,0.0,...,0.0,0.0,1.000000,0.0,1.000000,0.0,1.000000,2017-09-10 00:00:00,1.000000,0.0
std,0.464113,0.219232,0.341537,0.276249,0.258275,0.295604,0.227817,0.181631,0.0,0.0,...,0.0,0.0,0.203074,0.0,0.170572,0.0,0.259922,NaN,0.193691,0.0


In [ ]:
# strandart Scaling memebuat meanya mendekati 0
from sklearn.preprocessing import StandardScaler

In [81]:
standard_scaler = StandardScaler()

In [82]:
df_std_scaler = df_no_outliers.copy()
df_std_scaler[numerical_cols] = standard_scaler.fit_transform(df_no_outliers[numerical_cols])

In [83]:
df_std_scaler.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,previous_bookings_not_canceled,booking_changes,agent,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date,total_nights,customers
count,4.126400e+04,4.126400e+04,4.126400e+04,4.126400e+04,4.126400e+04,4.126400e+04,4.126400e+04,4.126400e+04,41264.0,41264.0,...,41264.0,41264.0,4.126400e+04,41264.0,4.126400e+04,41264.0,4.126400e+04,41264,4.126400e+04,41264.0
mean,2.892865e-17,-6.405630e-17,-7.430806e-14,1.074493e-16,-4.545931e-17,-1.102044e-17,6.599348e-17,4.967807e-17,0.0,0.0,...,0.0,0.0,2.066332e-18,0.0,-1.308677e-16,0.0,-6.750018e-17,2016-09-01 10:09:47.669639168,-2.935914e-17,0.0
min,-6.765204e-01,-1.047028e+00,-1.792838e+00,-1.782354e+00,-1.902764e+00,-1.662604e+00,-1.118901e+00,-1.595076e+00,0.0,0.0,...,0.0,0.0,-7.359256e-01,0.0,-2.585339e+00,0.0,-8.981884e-01,2014-11-18 00:00:00,-1.702360e+00,0.0
25%,-6.765204e-01,-8.144338e-01,-3.288507e-01,-7.950922e-01,-7.858739e-01,-8.732500e-01,-1.118901e+00,-9.833279e-01,0.0,0.0,...,0.0,0.0,-6.524614e-01,0.0,-6.568239e-01,0.0,-8.981884e-01,2016-03-22 00:00:00,-7.636478e-01,0.0
50%,-6.765204e-01,-3.104794e-01,-3.288507e-01,-1.369177e-01,3.317864e-02,2.886894e-02,-2.151477e-02,-3.715796e-01,0.0,0.0,...,0.0,0.0,-6.524614e-01,0.0,-1.091256e-01,0.0,3.842630e-01,2016-09-09 00:00:00,-2.942916e-01,0.0
75%,1.478152e+00,5.682104e-01,1.135136e+00,8.503442e-01,7.777718e-01,9.309878e-01,1.075871e+00,8.519170e-01,0.0,0.0,...,0.0,0.0,9.055360e-01,0.0,6.288528e-01,0.0,3.842630e-01,2017-03-06 00:00:00,6.444206e-01,0.0
max,1.478152e+00,3.514406e+00,1.135136e+00,1.837606e+00,1.969121e+00,1.720342e+00,3.270643e+00,3.910659e+00,0.0,0.0,...,0.0,0.0,4.188459e+00,0.0,3.277347e+00,0.0,2.949166e+00,2017-09-10 00:00:00,3.460557e+00,0.0
std,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,1.000012e+00,0.0,0.0,...,0.0,0.0,1.000012e+00,0.0,1.000012e+00,0.0,1.000012e+00,NaN,1.000012e+00,0.0


Kapan digunakanya?

In [84]:
from sklearn.preprocessing import RobustScaler

In [85]:
robust_scaler = RobustScaler()

In [87]:
df_robust = df_no_outliers.copy()
df_robust[numerical_cols] = robust_scaler.fit_transform(df_no_outliers[numerical_cols])

In [89]:
df_robust.describe()

,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,previous_bookings_not_canceled,booking_changes,agent,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,reservation_status_date,total_nights,customers
count,41264.000000,41264.000000,41264.000000,41264.000000,41264.000000,41264.000000,41264.000000,41264.000000,41264.0,41264.0,...,41264.0,41264.0,41264.000000,41264.0,41264.000000,41264.0,41264.000000,41264,41264.000000,41264.0
mean,0.313978,0.224555,0.224627,0.083211,-0.021219,-0.016001,0.009803,0.202469,0.0,0.0,...,0.0,0.0,0.418782,0.0,0.084878,0.0,-0.299632,2016-09-01 10:09:47.669639168,0.209004,0.0
min,0.000000,-0.532710,-1.000000,-1.000000,-1.238095,-0.937500,-0.500000,-0.666667,0.0,0.0,...,0.0,0.0,-0.053571,0.0,-1.926000,0.0,-1.000000,2014-11-18 00:00:00,-1.000000,0.0
25%,0.000000,-0.364486,0.000000,-0.400000,-0.523810,-0.500000,-0.500000,-0.333333,0.0,0.0,...,0.0,0.0,0.000000,0.0,-0.426000,0.0,-1.000000,2016-03-22 00:00:00,-0.333333,0.0
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,2016-09-09 00:00:00,0.000000,0.0
75%,1.000000,0.635514,1.000000,0.600000,0.476190,0.500000,0.500000,0.666667,0.0,0.0,...,0.0,0.0,1.000000,0.0,0.574000,0.0,0.000000,2017-03-06 00:00:00,0.666667,0.0
max,1.000000,2.766355,1.000000,1.200000,1.238095,0.937500,1.500000,2.333333,0.0,0.0,...,0.0,0.0,3.107143,0.0,2.634000,0.0,2.000000,2017-09-10 00:00:00,2.666667,0.0
std,0.464113,0.723261,0.683074,0.607749,0.639539,0.554257,0.455634,0.544893,0.0,0.0,...,0.0,0.0,0.641857,0.0,0.777810,0.0,0.779766,NaN,0.710201,0.0


## Korelation Matriks

In [90]:
# .corr()
df[numerical_cols].corr()

,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,total_nights,customers
is_canceled,1.000000,0.184772,0.088021,0.003735,0.001469,0.005317,0.060159,0.082886,0.081775,0.067355,...,0.051464,-0.052160,-0.093664,-0.000920,0.004461,0.127974,-0.184224,-0.120567,0.084059,0.100217
lead_time,0.184772,1.000000,0.139123,0.106216,0.101195,0.009852,0.235109,0.310088,0.140439,0.028630,...,0.005369,-0.078941,0.077005,0.080425,0.132151,0.023533,-0.086564,0.034213,0.318228,0.126681
arrival_date_year,0.088021,0.139123,1.000000,-0.499849,-0.514194,-0.010073,0.005127,0.003623,0.038573,0.041153,...,-0.054217,0.027255,0.008596,-0.001902,-0.027943,0.176063,-0.039816,0.064246,0.004607,0.050705
arrival_date_month,0.003735,0.106216,-0.499849,1.000000,0.995084,0.000827,0.027150,0.031778,0.027170,0.013659,...,0.007546,-0.021713,0.011751,0.020768,0.012763,0.103543,0.007330,0.049607,0.033754,0.031288
arrival_date_week_number,0.001469,0.101195,-0.514194,0.995084,1.000000,0.093641,0.026901,0.027842,0.024429,0.013464,...,0.007202,-0.020816,0.011919,0.019624,0.013847,0.098341,0.008959,0.046622,0.030735,0.029035
arrival_date_day_of_month,0.005317,0.009852,-0.010073,0.000827,0.093641,1.000000,-0.017801,-0.028216,-0.001143,0.015816,...,-0.008540,0.000150,0.006300,0.006122,0.006587,0.022597,0.009162,-0.001667,-0.027615,0.008120
stays_in_weekend_nights,0.060159,0.235109,0.005127,0.027150,0.026901,-0.017801,1.000000,0.555543,0.088232,0.028545,...,-0.020641,-0.056660,0.050297,0.158358,-0.031685,0.038926,-0.042938,0.032365,0.786258,0.087927
stays_in_week_nights,0.082886,0.310088,0.003623,0.031778,0.027842,-0.028216,0.555543,1.000000,0.095519,0.030458,...,-0.018788,-0.058513,0.085020,0.190033,0.001899,0.053285,-0.044327,0.037809,0.950575,0.095108
adults,0.081775,0.140439,0.038573,0.027170,0.024429,-0.001143,0.088232,0.095519,1.000000,0.023691,...,-0.042106,-0.120928,-0.048093,0.029570,-0.015746,0.248939,0.007780,0.112755,0.103930,0.804695
children,0.067355,0.028630,0.041153,0.013659,0.013464,0.015816,0.028545,0.030458,0.023691,1.000000,...,-0.019210,-0.029426,0.031304,0.042026,-0.020420,0.326302,0.036325,0.044590,0.033293,0.595117
